# Delivery Route Optimisation Using ILP and SA

IT5082 Optimization Methods

**Problem:**  Capacitated Vehicle Routing Problem (CVRP) <br>
**Exact method:** ILP <br>
**Heuristic:**  Simulated Annealing (SA) <br>
**Data:**  CVRPLIB benchmarks and the Amazon Last Mile 2021 routes<br>
**M L I U Meegahawala (MS26903538)  / M T M Ibjas (MS25942248)**

## Setup
Install the packages from requirements.txt

In [5]:
import itertools, json, math, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import milp, linprog, LinearConstraint, Bounds
from scipy.sparse import coo_matrix
import vrplib

# numba speedup the  SA decoder
try:
    from numba import njit
except ImportError:
    def njit(*args, **kwargs):
        return args[0] if args and callable(args[0]) else (lambda f: f)


SEED = 42
CONFIG = dict(
    exact_time_limit = 120,    # seconds per ILP solve
    exact_max_n      = 45,     # largest instance we try with the ILP
    n_seeds          = 10,     # SA runs per instance
    sa_iters_per_n   = 1500,   # SA iterations = this x number of customers
)

def sa_budget(inst):
    """SA iterations for an instance. The same for every seed, so the comparison is fair."""
    return CONFIG["sa_iters_per_n"] * inst["n"]


# Folders: raw input files, saved instances, figures. Created if they do not exist yet.
RAW, INST, FIG = Path("data/raw"), Path("data/instances"), Path("results/figures")
for p in (RAW, INST, FIG, Path("results")):
    p.mkdir(parents=True, exist_ok=True)

# Plot style for every figure: default size, light grid, no top and right border lines
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

# Fixed random seeds, so a re-run gives the same numbers
random.seed(SEED)
np.random.seed(SEED)

print("Config:", CONFIG)

Config: {'exact_time_limit': 120, 'exact_max_n': 45, 'n_seeds': 10, 'sa_iters_per_n': 1500}


## Load the CVRPLIB instances
CVRPLIB gives a proven optimal cost for each instance. We use it to check our ILP and to measure how far SA is from the best answer. <br>The `.vrp` files are added to `data/raw/.

In [12]:
CVRPLIB_OPTIMA = {                      # proven optimal costs, published on CVRPLIB
    "P-n16-k8": 450, "E-n22-k4": 375, "A-n32-k5": 784,
    "A-n33-k5": 661, "A-n60-k9": 1354, "A-n80-k10": 1763,
}

def find_instances():
    """Return the CVRPLIB instances that have a .vrp file in data/raw/."""
    found = [n for n in CVRPLIB_OPTIMA if (RAW / f"{n}.vrp").exists()]
    print(f"found {len(found)} instance files in {RAW}/")
    return found

## Convert to common instance format
ILP and SA read the same input. instance is a plain Python dict.
<br>Distances use CVRPLIB's straight-line distance rounded to the nearest integer.(euclidean_matrix)
<br>The depot is node `0`.

In [16]:
def euclidean_matrix(coords):
    """Distance between every pair of points, rounded to the nearest integer (CVRPLIB rule)."""
    pts = np.asarray(coords, dtype=float)
    d = np.sqrt(((pts[:, None, :] - pts[None, :, :]) ** 2).sum(-1))   # straight-line distances
    return np.rint(d).astype(int)


def parse_cvrplib(path, name=None):
    """Read one .vrp file and return it in to a dict."""
    raw = vrplib.read_instance(str(path))
    name = name or Path(path).stem
    demand = np.asarray(raw["demand"], dtype=int)
    coords = np.asarray(raw["node_coord"], dtype=float)
    return {
        "name": name, "source": "CVRPLIB", "n": len(demand) - 1,     # n = customers without the depot
        "coords": coords.tolist(), "demand": demand.tolist(),
        "capacity": int(raw["capacity"]), "depot": 0,
        "vehicles": int(name.rsplit("-k", 1)[1]),                    # fleet size is in the name A-n32-k5 -> 5
        "dist": euclidean_matrix(coords).tolist(),
        "cost_unit": "distance units", "known_opt": CVRPLIB_OPTIMA.get(name),
    }

def save_instance(inst):
    """Save an instance as JSON in data/instances/."""
    p = INST / f"{inst['name']}.json"
    p.write_text(json.dumps(inst))
    return p

def validate_instance(inst):
    """Valiidate the instances, so a bad instance can be removed"""
    d, D = np.asarray(inst["demand"]), np.asarray(inst["dist"])
    nm = inst["name"]
    assert inst["depot"] == 0, f"{nm}: depot must be node 0"
    assert D.shape == (len(d), len(d)), f"{nm}: matrix shape {D.shape} != {len(d)}"
    assert d[0] == 0, f"{nm}: depot has non-zero demand"
    assert d.max() <= inst["capacity"], f"{nm}: a single demand exceeds capacity"
    assert (np.diag(D) == 0).all() and (D >= 0).all(), f"{nm}: bad cost matrix"
    need = math.ceil(d.sum() / inst["capacity"])                     # fewest vehicles that could carry all demand
    assert need <= inst["vehicles"], f"{nm}: needs {need} vehicles but fleet is {inst['vehicles']}"
    return True


def instance_summary(instances):
    """summary table for report"""
    rows = []
    for i in instances:
        d = np.asarray(i["demand"])
        rows.append({"name": i["name"], "source": i["source"], "n": i["n"], "capacity": i["capacity"],
                     "total_demand": int(d.sum()), "vehicles": i["vehicles"],
                     "min_vehicles": math.ceil(d.sum() / i["capacity"]),
                     "tightness": round(d.sum() / (i["vehicles"] * i["capacity"]), 3),   # 1.0 = fleet is completely full
                     "known_opt": i["known_opt"], "cost_unit": i["cost_unit"]})
    return pd.DataFrame(rows).sort_values(["source", "n"]).reset_index(drop=True)
